In [ ]:
import time
from datetime import datetime, timezone
import dateparser
import pandas as pd
import requests
from bs4 import BeautifulSoup

START_ID = 12751        # aqui empiezan a poner bien los acentos, se quedo en el 34516
END_ID   = 39030        # ultimo publicado (al 2026-11-01)
DELAY    = 1.0          # para no ser tan atragantados
OUTPUT   = "./data/raw/unison_notas.parquet"
BASE_URL = "https://www.unison.mx/nota/?idnoti={}"
 
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}


session = requests.Session()
session.headers.update(HEADERS)
rows = []
for i in range(START_ID, END_ID + 1):
    try:
        r = session.get(BASE_URL.format(i), timeout=15)
        r.encoding = r.apparent_encoding  # fixes accents/ñ
        if r.ok:
            soup = BeautifulSoup(r.text, "html.parser")
            if not soup.find('h1',class_='page-entry-title'): # notas vacias (talvez las borraron)
                continue
            titular = soup.find('h1',class_='page-entry-title').text
            fecha = dateparser.parse(soup.find('div',class_='entry-content').find('p').text, languages=['es']).date().isoformat()
            contenido = soup.find('div',class_='entry-content').find('span').text.replace('\n\r','')
            autor = None
            if i >= 27456: # desde aqui empiezan a poner a quien escribio el articulo en la primera linea
                autor = contenido.split('\n')[0]
                contenido = "\n".join(contenido.split('\n')[1:])
            rows.append({
                "id": i,
                "headline": titular,
                "published": fecha,
                "text": contenido,
                "author": autor,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            })
            print(f"[{i}] OK   {rows[-1]['headline']!r}")
    except requests.RequestException as e:
        print(f"[{i}] ERROR {e}")

    # checkpoint every 25 pages so a crash doesn't lose everything
    if len(rows) % 25 == 0:
        pd.DataFrame(rows).to_parquet(OUTPUT, index=False)

    time.sleep(DELAY)

df = pd.DataFrame(rows)
df.to_parquet(OUTPUT, index=False)
print(f"\nSaved {len(df)} rows to {OUTPUT}")
df[["id", "published", "headline"]].sample(15)